In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pycaret.regression import *
from xgboost import XGBRegressor

In [11]:
df = pd.read_csv("C:\\Users\\ripa_\\Desktop\\Programing\\IndyCar_Project\\IndyCar\\datasets\\IndyCar_dataset_v22.csv")

In [12]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values("EventDate")

In [ ]:
print(df[["DriverID", "NormalizedPositionFinish", "DRFAvg"]].groupby("DriverID").head(3).head(30))

In [ ]:
df.head()

In [ ]:
#Pre-Qualy Non-ID Model
drop_cols = [
    "DriverName", "DriverID", "PositionStart", "TeamName", "TeamID", "CarEngine", "EngineID", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish"
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

In [ ]:
#Pre-Qualy ID Model
drop_cols = [
    "DriverName", "PositionStart", "StartVsField","TeamName", "CarEngine", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish",
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges",
    "AvgRaceSpeed", "FastestLapSpeed",
    "AvgStintLength", "StintVariance", "FirstStopLap", "FirstStopVSAvgFirstStop",
    "PittedCautions", "PitStops", "PittedCautionPCT", "TotalPitStops",
    "PitStrategy", #"PitStrategyID",
    "BestRacePosition", "WorstRacePosition", "LapsLed", "Top5Laps", "Top10Laps",
    "PositionVolatility", "position_lap1", "PositionsGainedLap1",
    "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency", "PaceDeg", "AvgPaceVSLeadPace",
    "AvgPitTime", "PitAvgTimeVSOverall",
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

In [13]:
#Post-Qualy Model
drop_cols = [
    "DriverName", "TeamName", "CarEngine", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish",
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges",
    "AvgRaceSpeed", "FastestLapSpeed",
    "AvgStintLength", "StintVariance", "FirstStopLap", "FirstStopVSAvgFirstStop",
    "PittedCautions", "PitStops", "PittedCautionPCT", "TotalPitStops",
    "PitStrategy", #"PitStrategyID",
    "BestRacePosition", "WorstRacePosition", "LapsLed", "Top5Laps", "Top10Laps",
    "PositionVolatility", "position_lap1", "PositionsGainedLap1",
    "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency", "PaceDeg", "AvgPaceVSLeadPace",
    "AvgPitTime", "PitAvgTimeVSOverall",
]

cutoff = df["EventDate"].quantile(0.95)
data = data.fillna(data.median(numeric_only=True))
data_unseen = data_unseen.fillna(data_unseen.median(numeric_only=True))
#data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
#data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

DriverElo                      -3.942621e-01
DriverTTElo                    -3.650209e-01
TeamElo                        -3.253071e-01
DriverTElo                     -2.409912e-01
TeamTElo                       -2.387766e-01
TeamID                         -1.217867e-01
EngineTTElo                    -7.356535e-02
EngineElo                      -6.092054e-02
EngineTElo                     -4.369402e-02
TeamTrackAvgPitStops           -1.667147e-02
TeamTrackAvgPittedCautionPCT   -1.015681e-02
TrackID                        -2.530634e-03
TrackAvgPitStops               -2.415214e-03
EraID                          -2.758440e-15
FieldSize                      -1.534615e-15
TotalRaceLaps                  -3.266062e-17
TrackLength                     1.124321e-16
EventTrackTypeID                1.406404e-16
TrackTypeAvgCautionLaps         4.856546e-04
TrackAvgSpeed                   5.292897e-04
FuelWindowEstimate              6.589211e-04
TrackTypeAvgCautions            8.715327e-04
TrackAvgCa

In [14]:
df = df.drop(columns=drop_cols)

In [15]:
print(df.columns.tolist())

['DriverID', 'Rookie', 'DRFAvg', 'DTAvg', 'DTTAvg', 'DNFRate', 'TDNFRate', 'DriverElo', 'DriverTElo', 'DriverTTElo', 'DriverRitmo', 'PositionStart', 'StartVsField', 'TeamID', 'TRP', 'TTP', 'TeamDNFRate', 'TeamElo', 'TeamTElo', 'TeamRitmo', 'TeamTrackAvgPitStops', 'TeamTrackAvgPittedCautionPCT', 'EngineID', 'EngineElo', 'EngineTElo', 'EngineTTElo', 'TrackID', 'EventTrackTypeID', 'TrackAvgCautions', 'TrackAvgCautionLaps', 'TrackTypeAvgCautions', 'TrackTypeAvgCautionLaps', 'TrackAvgSpeed', 'EraID', 'FieldSize', 'TotalRaceLaps', 'TrackLength', 'TrackAvgPitStops', 'FuelWindowEstimate', 'NormalizedPositionFinish']


In [16]:
exp = setup(
    data=data, 
    target="NormalizedPositionFinish", 
    session_id=123, 
    fold_strategy="timeseries",
    data_split_shuffle=False,
    fold_shuffle=False
)

,Description,Value
0,Session id,123
1,Target,NormalizedPositionFinish
2,Target type,Regression
3,Original data shape,"(5425, 40)"
4,Transformed data shape,"(5425, 40)"
5,Transformed train set shape,"(3797, 40)"
6,Transformed test set shape,"(1628, 40)"
7,Numeric features,39
8,Preprocess,True
9,Imputation type,simple


In [ ]:
compare_models()

In [34]:
rf = create_model('rf')
rf_tune = tune_model(rf)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2418,0.0801,0.2830,0.1041,0.1954,1.1235
1,0.2510,0.0880,0.2966,0.0367,0.2055,1.1473
2,0.2406,0.0805,0.2837,0.1147,0.1956,1.0578
3,0.2323,0.0776,0.2786,0.1451,0.1912,1.0092
4,0.2483,0.0852,0.2918,0.0618,0.2004,1.0724
5,0.2254,0.0752,0.2743,0.1794,0.1875,0.9962
6,0.2248,0.0759,0.2756,0.1522,0.1875,0.9025
7,0.2361,0.0787,0.2806,0.1374,0.1916,1.0057
8,0.2249,0.0692,0.2630,0.2381,0.1798,0.9791


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2385,0.0781,0.2795,0.1265,0.1927,1.0943
1,0.2427,0.0824,0.2870,0.0979,0.1982,1.0832
2,0.2392,0.0799,0.2827,0.1209,0.1942,1.0277
3,0.2303,0.0760,0.2758,0.1623,0.1888,0.9916
4,0.2336,0.0752,0.2743,0.1713,0.1881,0.9800
5,0.2217,0.0718,0.2680,0.2162,0.1835,0.9893
6,0.2239,0.0733,0.2707,0.1817,0.1842,0.8967
7,0.2309,0.0757,0.2751,0.1708,0.1868,0.9617
8,0.2218,0.0689,0.2625,0.2415,0.1791,0.9368


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [40]:
predict_model(rf_tune);
#predict_model(rf);
newpred2 = predict_model(rf_tune, data=data_unseen)
#newpred2 = predict_model(rf, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,0.2114,0.0648,0.2545,0.2780,0.1732,0.9238


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,0.2084,0.0653,0.2556,0.2732,0.1729,0.8768


In [35]:
gbr = create_model('gbr')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2473,0.0903,0.3004,-0.0093,0.2066,1.1749
1,0.2565,0.0946,0.3076,-0.0357,0.2126,1.1577
2,0.2461,0.0880,0.2967,0.0316,0.2049,1.0936
3,0.2331,0.0789,0.2809,0.1309,0.1927,1.0141
4,0.2428,0.0830,0.2881,0.0859,0.1972,1.0466
5,0.2258,0.0770,0.2775,0.1599,0.1914,1.0223
6,0.2241,0.0749,0.2737,0.1639,0.1859,0.9163
7,0.2339,0.0793,0.2817,0.1306,0.1911,0.9531
8,0.2242,0.0710,0.2664,0.2187,0.1818,0.9617


In [36]:
gbr_tune = tune_model(gbr)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2401,0.0790,0.2811,0.1163,0.1947,1.1217
1,0.2461,0.0838,0.2895,0.0822,0.1994,1.0857
2,0.2425,0.0812,0.2850,0.1066,0.1962,1.0671
3,0.2332,0.0762,0.2761,0.1604,0.1903,1.0367
4,0.2393,0.0776,0.2785,0.1453,0.1917,1.0275
5,0.2265,0.0738,0.2716,0.1954,0.1867,1.0389
6,0.2282,0.0735,0.2711,0.1792,0.1855,0.9412
7,0.2353,0.0768,0.2771,0.1587,0.1890,1.0100
8,0.2271,0.0705,0.2655,0.2240,0.1817,0.9812


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [43]:
#predict_model(gbr);
predict_model(gbr_tune);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,0.2176,0.0667,0.2584,0.2562,0.1770,0.9883


In [42]:
newpred3 = predict_model(gbr_tune, data=data_unseen)
#newpred3 = predict_model(gbr, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,0.2151,0.0674,0.2597,0.2497,0.1768,0.9432


In [17]:
cat = create_model('catboost')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2493,0.0872,0.2953,0.0251,0.2037,1.1698
1,0.2575,0.0935,0.3057,-0.0233,0.2102,1.1370
2,0.2438,0.0867,0.2945,0.0461,0.2031,1.0595
3,0.2333,0.0816,0.2857,0.1009,0.1959,1.0009
4,0.2441,0.0862,0.2936,0.0501,0.1998,0.9970
5,0.2197,0.0750,0.2738,0.1821,0.1876,0.9744
6,0.2265,0.0801,0.2830,0.1060,0.1903,0.8219
7,0.2348,0.0797,0.2824,0.1261,0.1917,0.9346
8,0.2222,0.0716,0.2676,0.2114,0.1817,0.9228


In [19]:
cat_tune = tune_model(cat)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2350,0.0802,0.2831,0.1036,0.1939,1.0486
1,0.2443,0.0840,0.2898,0.0801,0.1993,1.0464
2,0.2377,0.0809,0.2845,0.1097,0.1962,1.0428
3,0.2316,0.0766,0.2767,0.1566,0.1900,1.0069
4,0.2359,0.0774,0.2783,0.1470,0.1903,0.9792
5,0.2198,0.0727,0.2696,0.2069,0.1845,0.9637
6,0.2208,0.0726,0.2694,0.1895,0.1831,0.8938
7,0.2313,0.0777,0.2788,0.1482,0.1887,0.9366
8,0.2182,0.0684,0.2616,0.2466,0.1780,0.9052


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [46]:
predict_model(cat_tune);
#predict_model(cat);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,CatBoost Regressor,0.2059,0.0640,0.2531,0.2863,0.1712,0.8534


In [45]:
newpred5 = predict_model(cat_tune, data=data_unseen)
#newpred5 = predict_model(cat, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,CatBoost Regressor,0.2030,0.0654,0.2557,0.2723,0.1713,0.8178


In [18]:
lgbm = create_model('lightgbm')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2612,0.0970,0.3115,-0.0847,0.2148,1.2307
1,0.2619,0.0988,0.3143,-0.0816,0.2155,1.1366
2,0.2526,0.0921,0.3034,-0.0128,0.2095,1.1461
3,0.2326,0.0820,0.2864,0.0962,0.1953,0.9418
4,0.2438,0.0848,0.2913,0.0653,0.1990,0.9953
5,0.2258,0.0796,0.2822,0.1313,0.1934,0.9780
6,0.2232,0.0779,0.2791,0.1302,0.1880,0.7969
7,0.2346,0.0815,0.2855,0.1066,0.1923,0.8864
8,0.2266,0.0729,0.2699,0.1977,0.1835,0.9610


In [20]:
lgbm_tune = tune_model(lgbm)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2403,0.0784,0.2799,0.1238,0.1925,1.0820
1,0.2432,0.0834,0.2888,0.0866,0.1980,1.0253
2,0.2402,0.0818,0.2860,0.1002,0.1962,1.0281
3,0.2283,0.0757,0.2752,0.1658,0.1877,0.9553
4,0.2313,0.0739,0.2718,0.1863,0.1852,0.9385
5,0.2213,0.0722,0.2688,0.2120,0.1838,0.9835
6,0.2222,0.0722,0.2686,0.1942,0.1830,0.9004
7,0.2309,0.0763,0.2763,0.1633,0.1875,0.9555
8,0.2206,0.0683,0.2614,0.2477,0.1782,0.9261


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [49]:
predict_model(lgbm_tune);
#predict_model(lgbm);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,0.2074,0.0643,0.2536,0.2832,0.1719,0.8781


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [92]:
newpred5 = predict_model(lgbm_tune, data=data_unseen)
newpred5 = predict_model(lgbm, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,0.2039,0.0640,0.2529,0.2883,0.1701,0.8345


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,0.2158,0.0737,0.2715,0.1802,0.1829,0.8285


In [50]:
blend1 = blend_models([rf_tune, lgbm_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2384,0.0777,0.2787,0.1315,0.1919,1.0860
1,0.2422,0.0822,0.2867,0.0999,0.1973,1.0526
2,0.2393,0.0805,0.2838,0.1139,0.1948,1.0271
3,0.2291,0.0757,0.2750,0.1666,0.1880,0.9729
4,0.2322,0.0742,0.2724,0.1824,0.1862,0.9585
5,0.2212,0.0718,0.2680,0.2165,0.1834,0.9856
6,0.2229,0.0725,0.2693,0.1900,0.1834,0.8982
7,0.2307,0.0758,0.2754,0.1690,0.1870,0.9581
8,0.2211,0.0684,0.2615,0.2469,0.1784,0.9311


In [51]:
blend1_tune = tune_model(blend1)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2383,0.0777,0.2787,0.1316,0.1920,1.0865
1,0.2422,0.0822,0.2867,0.1003,0.1973,1.0550
2,0.2393,0.0805,0.2837,0.1147,0.1947,1.0272
3,0.2292,0.0757,0.2751,0.1664,0.1880,0.9744
4,0.2323,0.0743,0.2725,0.1818,0.1864,0.9602
5,0.2212,0.0718,0.2680,0.2166,0.1834,0.9859
6,0.2230,0.0726,0.2694,0.1895,0.1835,0.8981
7,0.2307,0.0758,0.2753,0.1693,0.1869,0.9584
8,0.2211,0.0684,0.2616,0.2466,0.1784,0.9316


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [56]:
#predict_model(blend1_tune);
predict_model(blend1);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2092,0.0643,0.2536,0.2836,0.1722,0.9003


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [57]:
#newpred5 = predict_model(blend1_tune, data=data_unseen)
newpred5 = predict_model(blend1, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2053,0.0644,0.2537,0.2837,0.1712,0.8541


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [52]:
blend2 = blend_models([rf_tune, lgbm_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2364,0.0776,0.2786,0.1322,0.1915,1.0711
1,0.2425,0.0823,0.2869,0.0988,0.1973,1.0496
2,0.2384,0.0802,0.2833,0.1172,0.1948,1.0315
3,0.2293,0.0754,0.2746,0.1694,0.1880,0.9828
4,0.2329,0.0749,0.2737,0.1749,0.1871,0.9642
5,0.2206,0.0718,0.2680,0.2164,0.1834,0.9781
6,0.2220,0.0723,0.2689,0.1928,0.1830,0.8962
7,0.2307,0.0762,0.2761,0.1645,0.1872,0.9506
8,0.2198,0.0682,0.2611,0.2494,0.1779,0.9219


In [53]:
blend2_tune = tune_model(blend2)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2376,0.0775,0.2785,0.1330,0.1916,1.0803
1,0.2423,0.0822,0.2867,0.1001,0.1972,1.0514
2,0.2389,0.0804,0.2835,0.1157,0.1948,1.0286
3,0.2291,0.0755,0.2748,0.1683,0.1879,0.9764
4,0.2324,0.0744,0.2728,0.1801,0.1865,0.9605
5,0.2209,0.0718,0.2679,0.2168,0.1834,0.9828
6,0.2226,0.0724,0.2691,0.1914,0.1832,0.8975
7,0.2307,0.0760,0.2756,0.1676,0.1870,0.9552
8,0.2206,0.0683,0.2613,0.2481,0.1782,0.9276


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [60]:
predict_model(blend2_tune);
#predict_model(blend2);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2087,0.0642,0.2533,0.2851,0.1719,0.8942


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [59]:
newpred6 = predict_model(blend2_tune, data=data_unseen)
#newpred6 = predict_model(blend2, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2049,0.0644,0.2537,0.2836,0.1710,0.8493


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [54]:
blend3 = blend_models([lgbm_tune, gbr_tune, rf_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2387,0.0778,0.2790,0.1296,0.1925,1.0973
1,0.2431,0.0824,0.2870,0.0979,0.1976,1.0628
2,0.2402,0.0805,0.2837,0.1147,0.1949,1.0399
3,0.2300,0.0754,0.2747,0.1689,0.1882,0.9930
4,0.2340,0.0749,0.2737,0.1749,0.1876,0.9800
5,0.2227,0.0722,0.2687,0.2121,0.1842,1.0029
6,0.2241,0.0726,0.2694,0.1894,0.1838,0.9115
7,0.2319,0.0758,0.2754,0.1691,0.1873,0.9748
8,0.2226,0.0688,0.2624,0.2419,0.1791,0.9468


In [61]:
blend3_tune = tune_model(blend3)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2386,0.0777,0.2788,0.1307,0.1923,1.0928
1,0.2427,0.0823,0.2868,0.0991,0.1974,1.0587
2,0.2398,0.0805,0.2837,0.1148,0.1948,1.0348
3,0.2296,0.0755,0.2747,0.1685,0.1881,0.9850
4,0.2332,0.0746,0.2731,0.1784,0.1870,0.9713
5,0.2220,0.0720,0.2684,0.2142,0.1839,0.9960
6,0.2236,0.0725,0.2693,0.1900,0.1836,0.9061
7,0.2313,0.0758,0.2753,0.1695,0.1871,0.9680
8,0.2220,0.0686,0.2620,0.2442,0.1788,0.9405


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [74]:
predict_model(blend3_tune);
#predict_model(blend3);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2105,0.0645,0.2540,0.2809,0.1728,0.9172


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [75]:
newpred = predict_model(blend3_tune, data=data_unseen)
#newpred = predict_model(blend3, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2069,0.0648,0.2546,0.2788,0.1721,0.8712


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [62]:
blend4 = blend_models([rf_tune, lgbm_tune, gbr_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2370,0.0776,0.2786,0.1321,0.1919,1.0831
1,0.2428,0.0823,0.2869,0.0984,0.1975,1.0576
2,0.2391,0.0802,0.2832,0.1178,0.1948,1.0395
3,0.2300,0.0753,0.2745,0.1701,0.1882,0.9957
4,0.2341,0.0752,0.2743,0.1713,0.1879,0.9791
5,0.2218,0.0720,0.2684,0.2143,0.1839,0.9927
6,0.2230,0.0723,0.2689,0.1925,0.1833,0.9064
7,0.2316,0.0761,0.2758,0.1665,0.1873,0.9647
8,0.2213,0.0685,0.2617,0.2459,0.1785,0.9359


In [63]:
blend4_tune = tune_model(blend4)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2369,0.0776,0.2785,0.1325,0.1916,1.0753
1,0.2427,0.0825,0.2871,0.0972,0.1974,1.0457
2,0.2389,0.0805,0.2837,0.1149,0.1950,1.0345
3,0.2292,0.0753,0.2743,0.1708,0.1878,0.9824
4,0.2329,0.0747,0.2734,0.1766,0.1869,0.9643
5,0.2210,0.0719,0.2682,0.2152,0.1836,0.9838
6,0.2223,0.0722,0.2686,0.1943,0.1830,0.9010
7,0.2310,0.0762,0.2760,0.1651,0.1873,0.9560
8,0.2203,0.0682,0.2612,0.2488,0.1780,0.9269


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [78]:
predict_model(blend4_tune);
#predict_model(blend4);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2082,0.0641,0.2531,0.2862,0.1717,0.8896


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [79]:
newpred = predict_model(blend4_tune, data=data_unseen)
#newpred = predict_model(blend4, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2047,0.0645,0.2539,0.2827,0.1710,0.8469


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [64]:
blend5 = blend_models([rf_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2360,0.0781,0.2795,0.1264,0.1920,1.0695
1,0.2427,0.0823,0.2869,0.0986,0.1977,1.0630
2,0.2377,0.0798,0.2826,0.1216,0.1946,1.0335
3,0.2304,0.0757,0.2752,0.1656,0.1887,0.9980
4,0.2343,0.0759,0.2755,0.1639,0.1887,0.9786
5,0.2204,0.0719,0.2681,0.2161,0.1835,0.9758
6,0.2222,0.0726,0.2694,0.1896,0.1833,0.8949
7,0.2309,0.0764,0.2764,0.1626,0.1874,0.9487
8,0.2195,0.0683,0.2614,0.2475,0.1781,0.9200


In [65]:
blend5_tune = tune_model(blend5)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2370,0.0779,0.2790,0.1293,0.1920,1.0814
1,0.2424,0.0821,0.2866,0.1006,0.1977,1.0726
2,0.2383,0.0797,0.2824,0.1229,0.1942,1.0304
3,0.2302,0.0758,0.2752,0.1654,0.1886,0.9945
4,0.2338,0.0754,0.2747,0.1688,0.1882,0.9790
5,0.2210,0.0717,0.2679,0.2173,0.1834,0.9825
6,0.2230,0.0728,0.2699,0.1866,0.1837,0.8956
7,0.2309,0.0760,0.2756,0.1675,0.1870,0.9552
8,0.2206,0.0685,0.2618,0.2453,0.1785,0.9284


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [81]:
#predict_model(blend5_tune);
predict_model(blend5);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2083,0.0641,0.2531,0.2860,0.1717,0.8877


In [82]:
#newpred = predict_model(blend5_tune, data=data_unseen)
newpred = predict_model(blend5, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2051,0.0650,0.2550,0.2768,0.1716,0.8462


In [66]:
blend6 = blend_models([lgbm_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2359,0.0780,0.2792,0.1281,0.1916,1.0609
1,0.2430,0.0831,0.2883,0.0899,0.1979,1.0344
2,0.2386,0.0808,0.2843,0.1109,0.1956,1.0346
3,0.2292,0.0753,0.2745,0.1702,0.1879,0.9793
4,0.2328,0.0750,0.2739,0.1733,0.1870,0.9570
5,0.2203,0.0721,0.2686,0.2131,0.1837,0.9729
6,0.2211,0.0721,0.2684,0.1954,0.1827,0.8963
7,0.2308,0.0767,0.2770,0.1592,0.1877,0.9453
8,0.2191,0.0681,0.2609,0.2506,0.1777,0.9151


In [67]:
blend6_tune = tune_model(blend6)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2378,0.0778,0.2790,0.1295,0.1917,1.0707
1,0.2430,0.0831,0.2883,0.0899,0.1978,1.0296
2,0.2393,0.0812,0.2849,0.1070,0.1957,1.0311
3,0.2285,0.0753,0.2744,0.1702,0.1875,0.9666
4,0.2319,0.0743,0.2726,0.1815,0.1859,0.9473
5,0.2207,0.0721,0.2685,0.2135,0.1836,0.9781
6,0.2216,0.0720,0.2684,0.1957,0.1828,0.8983
7,0.2308,0.0765,0.2765,0.1622,0.1875,0.9503
8,0.2199,0.0681,0.2610,0.2500,0.1778,0.9206


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [85]:
predict_model(blend6_tune);
#predict_model(blend6);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2069,0.0640,0.2531,0.2863,0.1714,0.8715


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [84]:
newpred = predict_model(blend6_tune, data=data_unseen)
#newpred = predict_model(blend6, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2034,0.0641,0.2532,0.2870,0.1701,0.8297


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [68]:
blend7 = blend_models([lgbm_tune, gbr_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2395,0.0781,0.2794,0.1270,0.1929,1.1005
1,0.2437,0.0827,0.2876,0.0945,0.1977,1.0535
2,0.2408,0.0809,0.2845,0.1097,0.1955,1.0462
3,0.2299,0.0753,0.2743,0.1710,0.1881,0.9939
4,0.2342,0.0749,0.2736,0.1751,0.1875,0.9801
5,0.2234,0.0726,0.2694,0.2083,0.1848,1.0103
6,0.2245,0.0724,0.2691,0.1916,0.1838,0.9195
7,0.2326,0.0760,0.2757,0.1669,0.1876,0.9816
8,0.2233,0.0690,0.2627,0.2402,0.1794,0.9524


In [69]:
blend7_tune = tune_model(blend7)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2398,0.0781,0.2794,0.1271,0.1925,1.0908
1,0.2432,0.0828,0.2878,0.0930,0.1976,1.0386
2,0.2402,0.0812,0.2850,0.1065,0.1957,1.0365
3,0.2289,0.0753,0.2744,0.1703,0.1877,0.9739
4,0.2326,0.0742,0.2723,0.1831,0.1861,0.9586
5,0.2222,0.0723,0.2689,0.2113,0.1842,0.9964
6,0.2232,0.0722,0.2686,0.1942,0.1833,0.9097
7,0.2316,0.0761,0.2758,0.1665,0.1874,0.9681
8,0.2218,0.0686,0.2618,0.2451,0.1786,0.9387


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [87]:
predict_model(blend7_tune);
#predict_model(blend7);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2094,0.0644,0.2538,0.2822,0.1724,0.9038


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [88]:
newpred = predict_model(blend7_tune, data=data_unseen)
#newpred = predict_model(blend7, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2058,0.0644,0.2538,0.2831,0.1713,0.8595


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [70]:
blend8 = blend_models([gbr_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2361,0.0782,0.2796,0.1257,0.1926,1.0814
1,0.2441,0.0830,0.2880,0.0917,0.1982,1.0640
2,0.2391,0.0803,0.2833,0.1171,0.1953,1.0527
3,0.2318,0.0758,0.2753,0.1650,0.1895,1.0206
4,0.2366,0.0768,0.2771,0.1538,0.1901,1.0012
5,0.2224,0.0724,0.2691,0.2101,0.1845,1.0000
6,0.2236,0.0724,0.2690,0.1921,0.1835,0.9157
7,0.2327,0.0766,0.2767,0.1611,0.1880,0.9719
8,0.2218,0.0688,0.2623,0.2426,0.1789,0.9412


In [71]:
blend8_tune = tune_model(blend8)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2358,0.0783,0.2798,0.1244,0.1925,1.0763
1,0.2440,0.0830,0.2881,0.0914,0.1982,1.0612
2,0.2388,0.0802,0.2833,0.1172,0.1953,1.0510
3,0.2317,0.0758,0.2754,0.1645,0.1895,1.0184
4,0.2364,0.0768,0.2772,0.1537,0.1901,0.9978
5,0.2220,0.0723,0.2690,0.2108,0.1844,0.9946
6,0.2231,0.0723,0.2689,0.1927,0.1834,0.9123
7,0.2324,0.0766,0.2768,0.1601,0.1880,0.9665
8,0.2211,0.0686,0.2620,0.2441,0.1787,0.9358


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [91]:
predict_model(blend8_tune);
#predict_model(blend8);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2101,0.0644,0.2538,0.2822,0.1726,0.9091


In [90]:
newpred = predict_model(blend8_tune, data=data_unseen)
#newpred = predict_model(blend8, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2071,0.0656,0.2561,0.2702,0.1727,0.8692


In [ ]:
save_model(blend9, "indycar_lgbm_cat_prequaly_model_v3")

In [93]:
save_model(lgbm_tune, "indycar_lgbm_postqualy_model_v4")

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['DriverID', 'Rookie', 'DRFAvg',
                                              'DTAvg', 'DTTAvg', 'DNFRate',
                                              'TDNFRate', 'DriverElo',
                                              'DriverTElo', 'DriverTTElo',
                                              'DriverRitmo', 'PositionStart',
                                              'StartVsField', 'TeamID', 'TRP',
                                              'TTP', 'TeamDNFRate', 'TeamElo',
                                              'TeamTElo', 'TeamRitmo',
                                              'TeamTrackAvgPitStops',
                                              'TeamTrackAvgP...
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=[],
                      